# Notebook de mineria de texto

Objetivo:
* Extraer la información necesaria para poder entrenar al modelo en los conceptos educativos que se manejará en el proyecto

## Importación de librerias necesarias

In [1]:
import fitz
import camelot
import pandas as pd
import re
from pathlib import Path
import json

print("Librerías importadas con éxito.")

c:\Users\joshu\Desktop\tesis\TESIS-CORA-OLORTEGUI\.venv\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


Librerías importadas con éxito.


## Constantes del proyecto

In [3]:
ZDP_CLEANUP_PATTERNS = re.compile(
    r"Didasc@lia: Didáctica y Educación\."
    r"|Michel Enrique Gamboa Graus"
    r"|ISSN 2224-2643"
    r"|Vol\. X\. Año 2019\. Número 4, Octubre-Diciembre"
    r"|Revista Didasc@lia: D&E\. Publicación del CEPUT - Las Tunas, CUBA"
    r"|LA ZONA DE DESARROLLO PRÓXIMO"
    r"|\d{1,2}\n?$"
)

BLOOM_CLEANUP_PATTERNS = re.compile(
    r"Reflexiones y experiencias investigativas para la innovación"
    r"|Revista Innovaciones Educativas / ISSN 2215-4132/Vol\. 25 / Número 38 / Enero - Junio, 2023"
    r"|Innovaciones\nEducativas"
    r"|GAMBOA SOLANO/GUEVARA MORA/\nMENA/UMAÑA MATA"
    r"|REVISTA INNOVACIONES EDUCATIVAS\nCorreo: innoveducativas@uned\.ac\.cr"
    r"|Artículo protegido por licencia Creative Commons"
    r"|\d{3}\n?$"
)

FLOW_CLEANUP_PATTERNS = re.compile(
    r"Flow: una perspectiva\ndicotómica"
    r"|Rosa M Abdón Ferré"
    r"|Trabajo Final de Grado de Criminología"
    r"|Dirigido por Sergi Rufí Cano"
    r"|Curso 2013-2014"
    r"|\d{1,2}\n?$"
)

# PARÁMETROS DE EXTRACCIÓN
BLOOM_TABLE_PAGE = 5

## Rutas del proyecto

In [4]:
# Ruta base del proyecto
BASE_DIR = Path().cwd().parent

# Rutas de datos RAW (los PDFs originales)
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
PDF_ZDP = DATA_RAW_DIR / "ZonaDeDesarrolloProximo.pdf"
PDF_BLOOM = DATA_RAW_DIR / "TaxonomiaDeBloom.pdf"
PDF_FLOW = DATA_RAW_DIR / "TeoriaDelFlow.pdf"

# Rutas de datos PROCESSED (donde guardaremos los resultados)
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed"
TXT_ZDP_CLEAN = DATA_PROCESSED_DIR / "zdp_clean.txt"
TXT_FLOW_CLEAN = DATA_PROCESSED_DIR / "flow_clean.txt"
JSON_BLOOM_TABLE = DATA_PROCESSED_DIR / "bloom_table.json"

print(f"Directorio Base: {BASE_DIR}")
print(f"Directorio de PDFs: {DATA_RAW_DIR}")
print(f"Directorio de Salida: {DATA_PROCESSED_DIR}")

Directorio Base: c:\Users\joshu\Desktop\tesis\TESIS-CORA-OLORTEGUI\src
Directorio de PDFs: c:\Users\joshu\Desktop\tesis\TESIS-CORA-OLORTEGUI\src\data\raw
Directorio de Salida: c:\Users\joshu\Desktop\tesis\TESIS-CORA-OLORTEGUI\src\data\processed


## Funciones Personalizadas

In [6]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extrae el texto crudo de todas las páginas de un archivo PDF.
    """
    doc = fitz.open(pdf_path)
    full_text = ""
    for page in doc:
        full_text += page.get_text() + "\n"
    doc.close()
    return full_text

def clean_pdf_text(text: str, noise_patterns: re.Pattern) -> str:
    """
    Limpia el texto crudo eliminando patrones de ruido (encabezados, pies de página)
    y corrigiendo saltos de línea.
    """
    # 1. Eliminar los patrones de ruido
    text = noise_patterns.sub("", text)
    
    # 2. Corregir saltos de línea con guion (ej. "conoci-miento")
    text = re.sub(r"-\n", "", text)
    
    # 3. Corregir saltos de línea innecesarios (unir párrafos)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    
    # 4. Opcional: eliminar espacios/líneas múltiples
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

def extract_table_from_pdf(pdf_path: str, page: int) -> pd.DataFrame:
    """
    Extrae la primera tabla encontrada en una página específica de un PDF
    usando Camelot.
    """
    print(f"[INFO] Extrayendo tabla de {pdf_path} (página {page})...")
    try:
        tables = camelot.read_pdf(str(pdf_path), pages=str(page), flavor='lattice')
        
        if tables:
            df = tables[0].df
            print("[SUCCESS] Tabla extraída con éxito.")
            return df
        else:
            print("[ERROR] No se encontraron tablas en la página.")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"[ERROR] Ocurrió un error al extraer la tabla: {e}")
        return pd.DataFrame()

print("Funciones definidas.")

Funciones definidas.


## Ejecucion

### Zona de Desarrollo Próximo

In [7]:
print("--- 1. Procesando el pdf de ZONA DE DESARROLLO PRÓXIMO ---")

raw_text_zdp = extract_text_from_pdf(PDF_ZDP)
clean_text_zdp = clean_pdf_text(raw_text_zdp, ZDP_CLEANUP_PATTERNS)

# Guardar el texto limpio
with open(TXT_ZDP_CLEAN, "w", encoding="utf-8") as f:
    f.write(clean_text_zdp)
    
print(f"[ZDP] Texto limpio guardado en {TXT_ZDP_CLEAN}")
print("\n--- Muestra del Texto Limpio (ZDP) ---")
print(clean_text_zdp[:500] + "...") # Muestra los primeros 500 caracteres

--- 1. Procesando el pdf de ZONA DE DESARROLLO PRÓXIMO ---


FileNotFoundError: no such file: 'c:\Users\joshu\Desktop\tesis\TESIS-CORA-OLORTEGUI\src\data\raw\ZonaDeDesarrolloProximo.pdf'